In [38]:
from pynq.overlays.base import BaseOverlay
import time
from datetime import datetime
base = BaseOverlay("base.bit")

In [39]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
unsigned int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODB
unsigned int read_gpio(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

/*
//void close_gpio(unsigned int pin){
unsigned int close_gpio(unsigned int pin){
    gpio gpio_pin = gpio_open(pin);
    gpio_close(gpio_pin);
    return 0;
}

//Function to reset all GPIO pins on the chosen PMOD
unsigned int reset_all_gpio(){
    for(unsigned int i = 0; i<= 7; ++i){
        close_gpio(i);
    }
    return 0;
}
*/

In [40]:
btns = base.btns_gpio

GREEN_PIN_ID = 5
RED_PIN_ID = 6
BLUE_PIN_ID = 7

LED_ON = 1
LED_OFF = 0

#to reset all GPIOs of PMODB

#sequential approach
'''
write_gpio(0, 0)
write_gpio(1, 0)
write_gpio(2, 0)
write_gpio(3, 0)
write_gpio(4, 0)
write_gpio(5, 0)
write_gpio(6, 0)
write_gpio(7, 0)
'''

for pinId in range(0, 8):
    print("clearing PIN %d" % (pinId))
    write_gpio(pinId, 0)

#reset_all_gpio() #doesn't work

clearing PIN 0
clearing PIN 1
clearing PIN 2
clearing PIN 3
clearing PIN 4
clearing PIN 5
clearing PIN 6
clearing PIN 7


In [41]:
import asyncio
from enum import Enum

#class FlashingLed(Enum):
DEFAULT = 0
freq10Hz_dc0p25 = 1
freq10Hz_dc0p75 = 2
freq100Hz_dc0p25 = 3
freq100Hz_dc0p75 = 4
freq1000Hz_dc0p25 = 5
freq1000Hz_dc0p75 = 6
MAX=7

task1LedControl = True
task2BtnControl= True
allLedOff = False
numButtonPress = 0

def gpioAllLedOn():
    write_gpio(GREEN_PIN_ID, LED_ON)
    write_gpio(RED_PIN_ID, LED_ON)
    write_gpio(BLUE_PIN_ID, LED_ON)

def gpioAllLedOff():
    write_gpio(GREEN_PIN_ID, LED_OFF)
    write_gpio(RED_PIN_ID, LED_OFF)
    write_gpio(BLUE_PIN_ID, LED_OFF)

async def flash_leds():
    #global task1LedControl, start
    #print("by Default white LED blinks every second %d"%numButtonPress)
    while task1LedControl:
        await asyncio.sleep(0.01)
        if numButtonPress == DEFAULT or numButtonPress >= MAX:
            #print("case 1")
            gpioAllLedOn()
            await asyncio.sleep(1)
            gpioAllLedOff()
            await asyncio.sleep(1)
            #duty cycle to be changed

        if numButtonPress == freq10Hz_dc0p25:
            #print("case 2")
            frequency = 10 #in Hz
            dutyCycle = 0.25 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
            oneCycleLen=1/frequency
            onTime = dutyCycle*oneCycleLen
            offTime = (1-dutyCycle)*oneCycleLen
            gpioAllLedOn()
            await asyncio.sleep(onTime)
            gpioAllLedOff()
            await asyncio.sleep(offTime)

        if numButtonPress == freq10Hz_dc0p75:
            #print("case 3")
            frequency = 10 #in Hz
            dutyCycle = 0.75 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
            oneCycleLen=1/frequency
            onTime = dutyCycle*oneCycleLen
            offTime = (1-dutyCycle)*oneCycleLen
            gpioAllLedOn()
            await asyncio.sleep(onTime)
            gpioAllLedOff()
            await asyncio.sleep(offTime)

        if numButtonPress == freq100Hz_dc0p25:
            #print("case 4")
            frequency = 100 #in Hz
            dutyCycle = 0.25 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
            oneCycleLen=1/frequency
            onTime = dutyCycle*oneCycleLen
            offTime = (1-dutyCycle)*oneCycleLen
            gpioAllLedOn()
            await asyncio.sleep(onTime)
            gpioAllLedOff()
            await asyncio.sleep(offTime)

        if numButtonPress == freq100Hz_dc0p75:
            #print("case 5")
            frequency = 100 #in Hz
            dutyCycle = 0.75 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
            oneCycleLen=1/frequency
            onTime = dutyCycle*oneCycleLen
            offTime = (1-dutyCycle)*oneCycleLen
            gpioAllLedOn()
            await asyncio.sleep(onTime)
            gpioAllLedOff()
            await asyncio.sleep(offTime)

        if numButtonPress == freq1000Hz_dc0p25:
            #print("case 6")
            frequency = 1000 #in Hz
            dutyCycle = 0.25 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
            oneCycleLen=1/frequency
            onTime = dutyCycle*oneCycleLen
            offTime = (1-dutyCycle)*oneCycleLen
            gpioAllLedOn()
            await asyncio.sleep(onTime)
            gpioAllLedOff()
            await asyncio.sleep(offTime)

        if numButtonPress == freq1000Hz_dc0p75:
            #print("case 7")
            frequency = 1000 #in Hz
            dutyCycle = 0.75 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
            oneCycleLen=1/frequency
            onTime = dutyCycle*oneCycleLen
            offTime = (1-dutyCycle)*oneCycleLen
            gpioAllLedOn()
            await asyncio.sleep(onTime)
            gpioAllLedOff()
            await asyncio.sleep(offTime)

        if allLedOff == True:
            print("shut off all")
            gpioAllLedOff()
            break

async def get_btns(_loop):
    global task1LedControl, task2BtnControl,allLedOff,numButtonPress
    #print("Press button 0 to try different freq and duty cycles, press button 1 to exit %d"%numButtonPress)
    while task2BtnControl:
        await asyncio.sleep(0.01)

        if btns[0].read() != 0:
            #print("case buttn")
            numButtonPress += 1
            if numButtonPress >= MAX:
                numButtonPress = DEFAULT
            #print("val %d"%numButtonPress)
            while btns[0].read():  #this avoids multiple counter increment
                time.sleep(0.1)
        if btns[1].read() != 0:
            task1LedControl = False
            task2BtnControl = False
            allLedOff=True
            _loop.stop() # exit both task
            print("All Led Turned OFF, Exiting program!")

            break

loop = asyncio.new_event_loop()
loop.create_task(flash_leds())
loop.create_task(get_btns(loop))
loop.run_forever()
loop.close()
print("End of Program.")

All Led Turned OFF, Exiting program!
End of Program.


In [ ]:
import asyncio

task1LedControl = True
task2BtnControl= True
flashingLed=True #by default LED will be flashing

greenLedOnly=False
redLedOnly=False
blueLedOnly=False
allLedOff=False

frequency = 100 #in Hz
dutyCycle = 0.95 # for example if one cycle is of 1Sec, it will stay on for 0.25sec and off for 0.75sec
oneCycleLen=1/frequency
onTime = dutyCycle*oneCycleLen
offTime = (1-dutyCycle)*oneCycleLen

async def flash_leds():
    #global task1LedControl, start
    print("by Default white LED blinks every second")
    while task1LedControl:
        if flashingLed == True:
            write_gpio(GREEN_PIN_ID, LED_ON)
            write_gpio(RED_PIN_ID, LED_ON)
            write_gpio(BLUE_PIN_ID, LED_ON)
            await asyncio.sleep(1)
            gpioAllLedOff()
            await asyncio.sleep(1)

        if greenLedOnly == True:
            #stopping everything & starting only GREEN
            write_gpio(GREEN_PIN_ID, LED_ON)
            write_gpio(RED_PIN_ID, LED_OFF)
            write_gpio(BLUE_PIN_ID, LED_OFF)      
            await asyncio.sleep(onTime)            
            gpioAllLedOff()
            await asyncio.sleep(offTime)
            #need to work on duty cycle

        if redLedOnly == True:
            #stopping everything & starting only RED
            write_gpio(RED_PIN_ID, LED_ON)
            write_gpio(GREEN_PIN_ID, LED_OFF)
            write_gpio(BLUE_PIN_ID, LED_OFF) 
            '''
            await asyncio.sleep(1)            
            gpioAllLedOff()
            await asyncio.sleep(1)
            '''
            await asyncio.sleep(onTime)            
            gpioAllLedOff()
            await asyncio.sleep(offTime)
        if blueLedOnly == True:
            #stopping everything & starting only BLUE
            write_gpio(BLUE_PIN_ID, LED_ON)
            write_gpio(RED_PIN_ID, LED_OFF)
            write_gpio(GREEN_PIN_ID, LED_OFF)
            '''
            await asyncio.sleep(1)            
            gpioAllLedOff()
            await asyncio.sleep(1)   
            '''
            await asyncio.sleep(onTime)            
            gpioAllLedOff()
            await asyncio.sleep(offTime)

        if allLedOff == True:        
            gpioAllLedOff()
            #await asyncio.sleep(1)    
            break

async def get_btns(_loop):
    global task1LedControl, task2BtnControl, flashingLed, greenLedOnly, redLedOnly, blueLedOnly, allLedOff
    while task2BtnControl:
        #await asyncio.sleep(0.01)
        await asyncio.sleep(0.1)
        if btns[0].read() != 0:
            greenLedOnly = True
            #print("only GREEN LED is ON")
            flashingLed=False
            redLedOnly=False
            blueLedOnly=False
            allLedOff=False
            #ideally should have been a bitmask which should have been reset and only green enum to be set

        if btns[1].read() != 0:
            redLedOnly = True
            #print("only RED LED is ON")
            flashingLed=False
            greenLedOnly=False
            blueLedOnly=False
            allLedOff=False

        if btns[2].read() != 0:
            blueLedOnly = True
            #print("only BLUE LED is ON")
            flashingLed=False
            greenLedOnly=False
            redLedOnly=False
            allLedOff=False

        if btns[3].read() != 0:
            allLedOff=True
            #stopping everything & exit            
            blueLedOnly = False
            flashingLed=False
            greenLedOnly=False
            redLedOnly=False
            #task1LedControl = False
            #task2BtnControl = False
            #_loop.stop() # exit both task
            print("All Led Turned OFF, Exiting program!")
            break

loop = asyncio.new_event_loop()
loop.create_task(flash_leds())
loop.create_task(get_btns(loop))
loop.run_forever()
loop.close()
print("End of Program.")

Task was destroyed but it is pending!
task: <Task pending name='Task-37' coro=<flash_leds() done, defined at /tmp/ipykernel_1017/1936125374.py:18> wait_for=<Future pending cb=[Task.__wakeup()]>>


by Default white LED blinks every second
All Led Turned OFF, Exiting program!
